# AutomixRouter - Training

This notebook demonstrates how to train the **AutomixRouter** (Automatic LLM Mixing Router).

## Overview

AutomixRouter implements cost-efficient routing with self-verification. It starts with a small model
and escalates to a larger model only when needed, based on verification of the response quality.

**Key Features**:
- POMDP-based routing strategy
- Self-consistency verification
- Cost-aware routing
- Automatic small/large model selection

**Routing Methods**:
1. **Threshold**: Simple threshold-based routing
2. **SelfConsistency**: Self-consistency check before escalation
3. **POMDP**: Partially Observable Markov Decision Process

## 1. Environment Setup

In [1]:
# Install required packages (for Colab)
!git clone https://github.com/ulab-uiuc/LLMRouter.git
%cd LLMRouter
!pip install -e .
!pip install transformers torch peft accelerate bitsandbytes

Cloning into 'LLMRouter'...
remote: Enumerating objects: 6017, done.
remote: Counting objects: 100% (172/172), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 6017 (delta 98), reused 87 (delta 81), pack-reused 5845 (from 2)
Receiving objects: 100% (6017/6017), 89.41 MiB | 58.06 MiB/s, done.
Resolving deltas: 100% (2946/2946), done.
Updating files: 100% (288/288), done.
/home/zhongjie/LLMRouter
Obtaining file:///home/zhongjie/LLMRouter
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llmrouter-lib (pyproject.toml) ... done
  Created wheel for llmrouter-lib: filename=llmrouter_lib-0.2.0-0.editable-py3-none-any.whl size=15709 sha256=25bea689e22427a883c02c524cf8900fc599db01e316d57542c77cafa23c906b
  Stored in directory: /tmp/pip-ephem-wheel-cache-e6eexm8o/wheels/82/4a/fd/59c4aec93c356c380d

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'
os.environ['ANTHROPIC_API_KEY'] = 'your-key'
# Or for multiple keys:
os.environ['API_KEYS'] = '["key1", "key2"]'

In [8]:
from llmrouter.models.automix import AutomixRouter, AutomixRouterTrainer
from llmrouter.utils import setup_environment

setup_environment()
print("Environment setup complete!")

Environment setup complete!


## 2. Configuration

AutomixRouter uses the following configuration parameters:

| Parameter | Description | Default |
|-----------|-------------|--------|
| `routing_method` | Routing strategy | "POMDP" |
| `num_bins` | Discretization bins | 8 |
| `small_model_cost` | Cost of small model | 1 |
| `large_model_cost` | Cost of large model | 50 |
| `verifier_cost` | Cost of verifier | 1 |

In [9]:
import yaml

CONFIG_PATH = "configs/model_config_train/automix.yaml"

with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

print("Current Configuration:")
print("=" * 50)
print(yaml.dump(config, default_flow_style=False))

Current Configuration:
data_path:
  llm_data: data/example_data/llm_candidates/default_llm.json
  llm_embedding_data: data/example_data/llm_candidates/default_llm_embeddings.json
  query_data_test: data/example_data/query_data/default_query_test.jsonl
  query_data_train: data/example_data/query_data/default_query_train.jsonl
  routing_data_test: data/example_data/routing_data/temp_test.jsonl
  routing_data_train: data/example_data/routing_data/temp.jsonl
hparam:
  cost_constraint: null
  device: cpu
  large_model_cost: 50
  max_workers: 1
  num_bins: 8
  num_inference_samples: 2
  routing_method: POMDP
  small_model_cost: 1
  verbose: true
  verifier_cost: 1
metric:
  weights:
    cost: 0
    llm_judge: 0
    performance: 1
model_path:
  ini_model_path: ''
  save_model_path: saved_models/automix/automix_model.pkl



## 3. Initialize Router

In [ ]:
router = AutomixRouter(yaml_path=CONFIG_PATH)

print("Router initialized successfully!")
print(f"Number of training samples: {len(router.routing_data_train)}")
print(f"Routing method: {config['hparam']['routing_method']}")

## 4. Training

In [11]:
trainer = AutomixRouterTrainer(router=router, device='cpu')

print("Trainer initialized!")

Trainer initialized!


In [12]:
print("Starting training...")
print("=" * 50)

trainer.train()

print("=" * 50)
print("Training completed!")

Starting training...
Training completed!


## 5. Model Verification

In [13]:
# Test prediction
test_query = {"query": "What is 2 + 2?"}
result = router.route_single(test_query)

print(f"Test query: {test_query['query']}")
print(f"Routed to: {result['model_name']}")

[DEBUG] get_llm_response_via_api: Using api_key=nvapi-rEKW...hJfP, base_url=https://integrate.api.nvidia.com/v1, model=qwen/qwen2.5-7b-instruct


100%|██████████| 1/1 [00:02<00:00,  2.65s/it]


[DEBUG] get_llm_response_via_api: Using api_key=nvapi-rEKW...hJfP, base_url=https://integrate.api.nvidia.com/v1, model=qwen/qwen2.5-7b-instruct


  0%|          | 0/1 [00:00<?, ?it/s]

[DEBUG] get_llm_response_via_api: Using api_key=nvapi-rEKW...hJfP, base_url=https://integrate.api.nvidia.com/v1, model=qwen/qwen2.5-7b-instruct


100%|██████████| 1/1 [00:01<00:00,  1.46s/it]

Test query: What is 2 + 2?
Routed to: qwen/qwen2.5-7b-instruct


## Summary

In this notebook, we:

1. **Loaded Configuration**: Set up AutomixRouter with YAML configuration
2. **Trained Model**: Learned POMDP-based routing policy
3. **Verified Model**: Tested routing with sample queries

**Key Takeaways**:
- AutomixRouter optimizes cost-performance tradeoff
- Self-verification helps avoid unnecessary escalations
- POMDP provides optimal policy under uncertainty

**Next Steps**:
- Use next part of notebook for inference

# AutomixRouter - Inference

This part of notebook demonstrates how to use a trained **AutomixRouter** for inference.

## 1. Environment Setup (optional)

## 2. Load Trained Router

In [14]:
CONFIG_PATH = "configs/model_config_train/automix.yaml"

router = AutomixRouter(yaml_path=CONFIG_PATH)
print("Router loaded!")

✅ Auto-detected small model: qwen2.5-7b-instruct (7.0B) -> qwen/qwen2.5-7b-instruct
✅ Auto-detected large model: mixtral-8x22b-instruct-v0.1 (141.0B) -> mistralai/mixtral-8x22b-instruct-v0.1
[DEBUG] init_providers: API key from OPENAI_API_KEY, length=70, first_10=nvapi-rEKW...
[DEBUG] get_llm_response_via_api: Using api_key=nvapi-rEKW...hJfP, base_url=https://integrate.api.nvidia.com/v1, model=qwen/qwen2.5-7b-instruct


  5%|▍         | 1/22 [00:01<00:33,  1.60s/it]

[DEBUG] get_llm_response_via_api: Using api_key=nvapi-rEKW...hJfP, base_url=https://integrate.api.nvidia.com/v1, model=qwen/qwen2.5-7b-instruct
[DEBUG] get_llm_response_via_api: Using api_key=nvapi-rEKW...hJfP, base_url=https://integrate.api.nvidia.com/v1, model=qwen/qwen2.5-7b-instruct


 14%|█▎        | 3/22 [00:02<00:10,  1.88it/s]

[DEBUG] get_llm_response_via_api: Using api_key=nvapi-rEKW...hJfP, base_url=https://integrate.api.nvidia.com/v1, model=qwen/qwen2.5-7b-instruct


...

[DEBUG] get_llm_response_via_api: Using api_key=nvapi-rEKW...hJfP, base_url=https://integrate.api.nvidia.com/v1, model=qwen/qwen2.5-7b-instruct
[DEBUG] get_llm_response_via_api: Using api_key=nvapi-rEKW...hJfP, base_url=https://integrate.api.nvidia.com/v1, model=qwen/qwen2.5-7b-instruct


 95%|█████████▌| 21/22 [06:33<00:18, 18.57s/it]

[DEBUG] get_llm_response_via_api: Using api_key=nvapi-rEKW...hJfP, base_url=https://integrate.api.nvidia.com/v1, model=qwen/qwen2.5-7b-instruct
[DEBUG] get_llm_response_via_api: Using api_key=nvapi-rEKW...hJfP, base_url=https://integrate.api.nvidia.com/v1, model=qwen/qwen2.5-7b-instruct


100%|██████████| 22/22 [06:52<00:00, 18.75s/it]

✅ MetaRouter initialized successfully (YAML + data loaded).
Router loaded!


## 3. Query Routing

In [15]:
EXAMPLE_QUERIES = [
    {"query": "What is 2 + 2?"},  # Simple - small model
    {"query": "Solve: 3x^2 - 7x + 2 = 0"},  # Medium complexity
    {"query": "Prove the Riemann Hypothesis."},  # Complex - large model
]

print("Routing Results:")
print("=" * 60)

for i, query in enumerate(EXAMPLE_QUERIES, 1):
    result = router.route_single(query)
    print(f"{i}. {query['query'][:50]}...")
    print(f"   Routed to: {result['model_name']}")

Routing Results:
[DEBUG] get_llm_response_via_api: Using api_key=nvapi-rEKW...hJfP, base_url=https://integrate.api.nvidia.com/v1, model=qwen/qwen2.5-7b-instruct


100%|██████████| 1/1 [00:00<00:00,  5.12it/s]


[DEBUG] get_llm_response_via_api: Using api_key=nvapi-rEKW...hJfP, base_url=https://integrate.api.nvidia.com/v1, model=qwen/qwen2.5-7b-instruct


  0%|          | 0/1 [00:00<?, ?it/s]

[DEBUG] get_llm_response_via_api: Using api_key=nvapi-rEKW...hJfP, base_url=https://integrate.api.nvidia.com/v1, model=qwen/qwen2.5-7b-instruct


...

[DEBUG] get_llm_response_via_api: Using api_key=nvapi-rEKW...hJfP, base_url=https://integrate.api.nvidia.com/v1, model=qwen/qwen2.5-7b-instruct


  0%|          | 0/1 [00:00<?, ?it/s]

[DEBUG] get_llm_response_via_api: Using api_key=nvapi-rEKW...hJfP, base_url=https://integrate.api.nvidia.com/v1, model=qwen/qwen2.5-7b-instruct


100%|██████████| 1/1 [00:05<00:00,  5.91s/it]

3. Prove the Riemann Hypothesis....
   Routed to: qwen/qwen2.5-7b-instruct


## 4. File-Based Inference

Load queries from a file and save results.

In [16]:
import json

# Load queries from a JSONL file
def load_queries_from_file(file_path):
    """Load queries from a JSONL file."""
    queries = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                queries.append(json.loads(line))
    return queries

# Save results to a JSONL file
def save_results_to_file(results, output_path):
    """Save routing results to a JSONL file."""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        for result in results:
            f.write(json.dumps(result, ensure_ascii=False) + '\n')
    print(f"Results saved to: {output_path}")

# Example: Load from default query file
QUERY_FILE = "data/example_data/query_data/default_query_test.jsonl"
OUTPUT_FILE = "outputs/automix_router_results.jsonl"

if os.path.exists(QUERY_FILE):
    # Load queries
    file_queries = load_queries_from_file(QUERY_FILE)
    print(f"Loaded {len(file_queries)} queries from: {QUERY_FILE}")
    
    # Route queries
    file_results = router.route_batch(batch=file_queries[:10])
    print(f"Routed {len(file_results)} queries")
    
    # Save results
    save_results_to_file(file_results, OUTPUT_FILE)
    
    # Show sample results
    print(f"\nSample results:")
    for i, result in enumerate(file_results[:3], 1):
        print(f"  {i}. {result.get('query', '')[:40]}... -> {result['model_name']}")
else:
    print(f"Query file not found: {QUERY_FILE}")
    print("Create a JSONL file with format: {\"query\": \"Your question\"}")

Loaded 706 queries from: data/example_data/query_data/default_query_test.jsonl
[DEBUG] get_llm_response_via_api: Using api_key=nvapi-rEKW...hJfP, base_url=https://integrate.api.nvidia.com/v1, model=qwen/qwen2.5-7b-instruct


100%|██████████| 1/1 [00:00<00:00,  4.79it/s]


[DEBUG] get_llm_response_via_api: Using api_key=nvapi-rEKW...hJfP, base_url=https://integrate.api.nvidia.com/v1, model=qwen/qwen2.5-7b-instruct


  0%|          | 0/1 [00:00<?, ?it/s]

[DEBUG] get_llm_response_via_api: Using api_key=nvapi-rEKW...hJfP, base_url=https://integrate.api.nvidia.com/v1, model=qwen/qwen2.5-7b-instruct


...

[DEBUG] get_llm_response_via_api: Using api_key=nvapi-rEKW...hJfP, base_url=https://integrate.api.nvidia.com/v1, model=qwen/qwen2.5-7b-instruct


100%|██████████| 1/1 [00:17<00:00, 17.75s/it]


[DEBUG] get_llm_response_via_api: Using api_key=nvapi-rEKW...hJfP, base_url=https://integrate.api.nvidia.com/v1, model=mistralai/mixtral-8x22b-instruct-v0.1


100%|██████████| 1/1 [00:00<00:00,  3.35it/s]

Routed 10 queries
Results saved to: outputs/automix_router_results.jsonl

Sample results:
  1. Q: There are 4 houses in a row, numbered... -> mistralai/mixtral-8x22b-instruct-v0.1
  2. Q: There are 3 houses in a row, numbered... -> qwen/qwen2.5-7b-instruct
  3. Q: There are 3 houses in a row, numbered... -> mistralai/mixtral-8x22b-instruct-v0.1


## Summary

AutomixRouter intelligently routes:
- Simple queries to small, cheap models
- Complex queries to large, capable models
- Uses self-verification to validate routing decisions